# [267] Virtual team 404

**Full names:**
- Marius Aleksander Brun (stud ID: 109518)
- Thomas Mogstad Barth (stud ID: 172010)
- Amund Dahlmo Berge (stud ID: 109546)

**Kaggle team name:** [267] Virtual team 404

In [ ]:
import shutil
import numpy as np
import pandas as pd
import xgboost as xgb

In [2]:
H = 168

def read_table(path, hourly=False):
    table = pd.read_csv(path)
    table['date'] = pd.to_datetime(table['date'], utc=True).dt.tz_localize(None)
    table = table.set_index('date')
    return table.resample('h').mean() if hourly else table

train = pd.read_csv('data/kernel/Unit_commitment_decisions.csv')
train['starttime'] = pd.to_datetime(train['starttime'], utc=True).dt.tz_localize(None)
target_columns = pd.read_csv('sample_submission.csv', nrows=1).columns[1:]
test = pd.read_csv('prediction_mapping.csv')
test['starttime'] = pd.to_datetime(test['starttime'], utc=True).dt.tz_localize(None)

price = read_table('data/kernel/Historical_day_ahead_price_2015_2025.csv', hourly=True)['Day-ahead price (EUR/MWh)']
inflow = read_table('data/kernel/Historical_inflow_1958_2025.csv')
inflow = inflow[inflow.index >= '2014-12-01'].resample('h').mean()
volume = read_table('data/kernel/Historical_volume_2015_2024.csv')
water_value = read_table('data/kernel/Synthetic_water_value_2015_2024.csv')
min_flow = read_table('data/extended/Constraint_min_flow.csv', hourly=True)
min_volume = read_table('data/extended/Constraint_min_volume.csv', hourly=True)

In [3]:
upstream = {
    'Hogga': 'Bandak', 'Tokke': 'Vinjevatn', 'Lio': 'Byrtevatn', 'Byrte': 'Botnedalsvatn',
    'Vinje': 'Vaamarvatn', 'Haukeli': 'Vatjern', 'Songa': 'Songavatn', 'Kjela': 'Foersvatn',
    'Vesle Kjela': 'Kjelavatn',
}

def weekly_window(table, starts):
    idx = table.index.get_indexer(starts)
    assert (idx >= 0).all()
    return table.to_numpy(dtype=np.float32)[idx[:, None] + np.arange(H)[None, :]]

def make_features(starts):
    starts = pd.DatetimeIndex(starts)
    n = len(starts)
    rep = lambda a: np.repeat(a, H)
    P = weekly_window(price.to_frame(), starts)[:, :, 0]
    I = weekly_window(inflow, starts)
    V = volume.reindex(starts).to_numpy(dtype=np.float32)
    W = water_value.reindex(starts).to_numpy(dtype=np.float32)
    W_end = water_value.reindex(starts + pd.Timedelta(days=7)).to_numpy(dtype=np.float32)
    hours = pd.DatetimeIndex((starts.values[:, None] + np.arange(H)[None, :] * np.timedelta64(1, 'h')).ravel())

    f = pd.DataFrame({
        't': np.tile(np.arange(H), n), 'hour': hours.hour, 'dow': hours.dayofweek,
        'month': hours.month, 'doy': hours.dayofyear,
        'price': P.ravel(), 'price_week_mean': rep(P.mean(1)), 'price_week_std': rep(P.std(1)),
        'price_week_min': rep(P.min(1)), 'price_week_max': rep(P.max(1)),
    })
    f['price_rel'] = f['price'] - f['price_week_mean']
    f['price_rank'] = (P.argsort(1).argsort(1) / (H - 1)).ravel()
    f['price_day_mean'] = np.repeat(P.reshape(n, 7, 24).mean(2), 24, axis=1).ravel()
    f['price_prev'] = np.concatenate([P[:, :1], P[:, :-1]], axis=1).ravel()
    f['price_next'] = np.concatenate([P[:, 1:], P[:, -1:]], axis=1).ravel()
    for j, name in enumerate(inflow.columns):
        f[f'inflow_{name}'] = I[:, :, j].ravel()
    f['inflow_total'] = I.sum(2).ravel()
    for j, name in enumerate(volume.columns):
        f[f'vol_{name}'] = rep(V[:, j])
    for j, name in enumerate(water_value.columns):
        f[f'wv_{name}'] = rep(W[:, j])
    for j, name in enumerate(water_value.columns):
        f[f'wv_end_{name}'] = rep(W_end[:, j])

    margins = {}
    for plant, res in upstream.items():
        margins[f'margin_{plant}'] = f['price'] - f[f'wv_{res}']
        margins[f'margin_end_{plant}'] = f['price'] - f[f'wv_end_{res}']
        margins[f'margin_end_wk_{plant}'] = f['price_week_mean'] - f[f'wv_end_{res}']

    MF = weekly_window(min_flow, starts)
    MV = weekly_window(min_volume, starts)
    V0 = volume.reindex(starts)
    extra = {}
    for j, river in enumerate(min_flow.columns):
        extra[f'minflow_{river}'] = MF[:, :, j].ravel()
        extra[f'minflow_{river}_wkmax'] = rep(MF[:, :, j].max(1))
    for j, name in enumerate(min_volume.columns):
        v0 = V0[name].to_numpy(dtype=np.float32)
        extra[f'minvol_{name}'] = MV[:, :, j].ravel()
        extra[f'volgap_{name}'] = rep(v0) - MV[:, :, j].ravel()
        extra[f'volgap_{name}_wkmin'] = rep(v0 - MV[:, :, j].max(1))
    return pd.concat([f, pd.DataFrame(margins), pd.DataFrame(extra)], axis=1)

In [ ]:
X = make_features(train['starttime'])
X_test = make_features(test['starttime'])
y = train[target_columns].to_numpy(dtype=np.int8).reshape(-1, 14, H).transpose(0, 2, 1).reshape(-1, 14)

device = 'cuda' if shutil.which('nvidia-smi') else 'cpu'

model = xgb.XGBClassifier(
    n_estimators=368, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8,
    min_child_weight=100, reg_lambda=20, tree_method='hist', device=device, random_state=42,
)
model.fit(X, y)

wide = model.predict_proba(X_test).reshape(-1, H, 14).transpose(0, 2, 1).reshape(len(test), -1)
submission = pd.DataFrame(wide, columns=target_columns)
submission.insert(0, 'Run No', test['Run No'].to_numpy())
submission.to_csv('2026-11-05_xgb_gridsearch_cv0.9810.csv', index=False)


c:\Programmer\tdt4173\.venv\Lib\site-packages\xgboost\core.py:569: UserWarning: [14:27:06] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
